# EcoShield AI — Ortak Split ve Preprocessing Tabanı

Bu notebook, bütün model deneylerinin kullanacağı ortak veri tabanını hazırlar.

- IEEE-CIS train dosyaları `TransactionID` üzerinden left join ile birleştirilir.
- `%70 train / %15 validation / %15 test` stratified ortak split oluşturulur.
- Kolon eleme kararları yalnızca train splitinden öğrenilir.
- Temizlenmiş ortak veri bir kez Parquet olarak kaydedilir.
- Split indeksleri, target/ID dizileri, kalite raporları ve feature şeması saklanır.

Modele özel preprocessing cache'leri burada topluca üretilmez. Görev 2 veya Görev 3 bir profile ilk kez ihtiyaç duyduğunda ortak Parquet'ten bir kez oluşturulur; sonraki çalıştırmalarda hazır cache doğrudan yüklenir.

> Bu notebook model eğitmez, threshold seçmez, Kaggle submission üretmez ve resmî etiketsiz test dosyalarını kullanmaz.

## 1. Çalışma modu

`QUICK_MODE=True`, akışı küçük veriyle kontrol etmek içindir. Kalıcı ortak taban için notebook `QUICK_MODE=False` ile çalıştırılmalıdır. Quick çıktıları nihai dosyaların üzerine yazılmaz.

In [1]:
QUICK_MODE = False
QUICK_ROWS = 150_000

print("QUICK_MODE:", QUICK_MODE)
print("Okunacak transaction satırı:", QUICK_ROWS if QUICK_MODE else "TAM VERİ")

QUICK_MODE: False
Okunacak transaction satırı: TAM VERİ


## 2. Ortak yardımcı modülü yükleme

Tekrar kullanılabilir kod `notebooks/common_preprocessing.py` içinde tutulur. Sonraki notebooklar aynı split, şema ve cache fonksiyonlarını buradan import eder.

In [2]:
import importlib.util
import json
import sys
from pathlib import Path

required_packages = {
    "joblib": "joblib",
    "numpy": "numpy",
    "pandas": "pandas",
    "psutil": "psutil",
    "pyarrow": "pyarrow",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm",
}
missing_packages = [
    pip_name
    for import_name, pip_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]
if missing_packages:
    raise ModuleNotFoundError(
        "Eksik paketler: " + ", ".join(missing_packages)
        + ". Önce pip install -r requirements.txt çalıştırıp kernel'i yeniden başlatın."
    )

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "common_preprocessing.py").exists():
    candidate = NOTEBOOK_DIR / "notebooks"
    if (candidate / "common_preprocessing.py").exists():
        NOTEBOOK_DIR = candidate
if not (NOTEBOOK_DIR / "common_preprocessing.py").exists():
    raise FileNotFoundError("notebooks/common_preprocessing.py bulunamadı.")
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from common_preprocessing import (
    analyze_common_data,
    build_project_paths,
    find_project_root,
    load_and_merge_train,
    save_cache_manifest,
    save_common_artifacts,
)

PROJECT_ROOT = find_project_root(Path.cwd())
PATHS = build_project_paths(PROJECT_ROOT)

print("Proje kökü:", PROJECT_ROOT)
print("Ortak modül:", NOTEBOOK_DIR / "common_preprocessing.py")

Proje kökü: C:\Users\pc\Desktop\YZTA-Bootcamp-2026
Ortak modül: c:\Users\pc\Desktop\YZTA-Bootcamp-2026\notebooks\common_preprocessing.py


c:\Users\pc\anaconda3\envs\torchcuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Ham train dosyalarını okuma ve birleştirme

CSV dosyaları chunk halinde okunur. Progress bar işlenen satırı, yüzdeyi, geçen süreyi ve ETA'yı; postfix RAM kullanımını gösterir. Join anahtarları ve target doğrulanır.

In [3]:
merged_df = load_and_merge_train(
    PATHS,
    quick_rows=QUICK_ROWS if QUICK_MODE else None,
)

print("Birleşik veri shape:", merged_df.shape)
display(merged_df["isFraud"].value_counts().sort_index().to_frame("count"))


BAŞLADI: IEEE-CIS train dosyalarını okuma


train_transaction.csv satır sayımı: 100%|██████████| 683M/683M [00:00<00:00, 2.52GB/s]
train_transaction.csv okunuyor: 100%|██████████| 590540/590540 [00:06<00:00, 85558.77 satır/s, ram_gb=1.98]


train_transaction.csv: 590,540 satır, 394 kolon


train_identity.csv satır sayımı: 100%|██████████| 26.5M/26.5M [00:00<00:00, 1.85GB/s]
train_identity.csv okunuyor: 100%|██████████| 144233/144233 [00:00<00:00, 456311.00 satır/s, ram_gb=2.01]


train_identity.csv: 144,233 satır, 41 kolon
TAMAMLANDI: IEEE-CIS train dosyalarını okuma | geçen süre: 7.8 sn | RAM: 2.02 GB

BAŞLADI: Transaction ve identity left join
TAMAMLANDI: Transaction ve identity left join | geçen süre: 0.2 sn | RAM: 2.22 GB
Birleşik veri shape: (590540, 434)


,count
isFraud,
0,569877
1,20663


## 4. Ortak split, veri kalite raporları ve feature şeması

Yüksek missing ve sabit kolonlar splitten sonra yalnızca train verisi üzerinden belirlenir. Validation ve test kolon kararlarına dahil edilmez.

In [4]:
common_data = analyze_common_data(merged_df)

display(common_data.quality_summary)
display(common_data.split_summary)
display(common_data.missing_report.head(30))

print("Kullanılabilir feature:", len(common_data.feature_columns))
print("Numeric feature:", len(common_data.numeric_columns))
print("Kategorik feature:", len(common_data.categorical_columns))
print("Yüksek missing nedeniyle çıkarılan:", common_data.high_missing_columns)
print("Sabit olduğu için çıkarılan:", common_data.constant_columns)


BAŞLADI: %70/%15/%15 stratified split oluşturma
TAMAMLANDI: %70/%15/%15 stratified split oluşturma | geçen süre: 0.3 sn | RAM: 2.19 GB

BAŞLADI: Train splitinde feature kalite kontrolleri
TAMAMLANDI: Train splitinde feature kalite kontrolleri | geçen süre: 1.2 sn | RAM: 3.52 GB

BAŞLADI: Veri kalitesi raporlarını hazırlama


Kardinalite hesaplanıyor: 100%|██████████| 434/434 [00:01<00:00, 299.37 kolon/s]


TAMAMLANDI: Veri kalitesi raporlarını hazırlama | geçen süre: 1.9 sn | RAM: 4.86 GB


,metric,value
0,row_count,590540.00000
1,column_count,434.00000
2,duplicate_row_count,0.00000
3,fraud_count,20663.00000
4,normal_count,569877.00000
5,fraud_ratio,0.03499


,split,row_count,row_ratio,fraud_count,fraud_ratio
0,train,413378,0.70,14464,0.034990
1,validation,88581,0.15,3100,0.034996
2,test,88581,0.15,3099,0.034985


,column,dtype,missing_count,missing_ratio
0,id_24,float64,585793,0.991962
1,id_25,float64,585408,0.991310
2,id_07,float64,585385,0.991271
3,id_08,float64,585385,0.991271
4,id_21,float64,585381,0.991264
5,id_26,float64,585377,0.991257
6,id_22,float64,585371,0.991247
7,id_23,str,585371,0.991247
8,id_27,str,585371,0.991247
9,dist2,float64,552913,0.936284


Kullanılabilir feature: 423
Numeric feature: 381
Kategorik feature: 42
Yüksek missing nedeniyle çıkarılan: ['id_07', 'id_08', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27']
Sabit olduğu için çıkarılan: []


## 5. Ortak tabanı kaydetme

Aşağıdaki dosyalar oluşturulur:

- `data/processed/common/common_feature_base.parquet`
- `outputs/splits/common_split_indices.npz`
- `outputs/splits/common_split_targets_ids.npz`
- `outputs/metadata/common_feature_schema.json`
- veri kalitesi ve split özet raporları.

Bu dosyalar Görev 2–7 boyunca ortak veri sözleşmesidir.

In [5]:
artifacts = {
    "common": save_common_artifacts(
        common_data,
        PATHS,
        quick_mode=QUICK_MODE,
    )
}
manifest_path = save_cache_manifest(
    PATHS,
    artifacts,
    quick_mode=QUICK_MODE,
)

print("Kaydedilen ortak çıktılar:")
for name, path in artifacts["common"].items():
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f" - {name}: {path.relative_to(PROJECT_ROOT)} ({size_mb:.2f} MB)")
print(" - manifest:", manifest_path.relative_to(PROJECT_ROOT))


BAŞLADI: Ortak Parquet tabanını kaydetme
TAMAMLANDI: Ortak Parquet tabanını kaydetme | geçen süre: 4.4 sn | RAM: 2.22 GB
Kaydedilen ortak çıktılar:
 - base: data\processed\common\common_feature_base.parquet (58.30 MB)
 - split: outputs\splits\common_split_indices.npz (0.86 MB)
 - labels: outputs\splits\common_split_targets_ids.npz (0.89 MB)
 - schema: outputs\metadata\common_feature_schema.json (0.01 MB)
 - manifest: outputs\metadata\common_cache_manifest.json


## 6. Kayıt sonrası doğrulama

Ortak Parquet, split, target/ID ve şema dosyaları yeniden açılarak satır sayıları, indeks kapsamı ve kolon sırası doğrulanır.

In [6]:
import numpy as np
import pandas as pd

common_artifacts = artifacts["common"]
saved_base = pd.read_parquet(common_artifacts["base"])
saved_split = np.load(common_artifacts["split"])
saved_labels = np.load(common_artifacts["labels"])
with common_artifacts["schema"].open(encoding="utf-8") as file:
    saved_schema = json.load(file)

saved_train_idx = saved_split["train_idx"]
saved_val_idx = saved_split["val_idx"]
saved_test_idx = saved_split["test_idx"]
all_saved_indices = np.concatenate([
    saved_train_idx,
    saved_val_idx,
    saved_test_idx,
])

assert len(saved_base) == len(merged_df)
assert len(all_saved_indices) == len(saved_base)
assert len(np.unique(all_saved_indices)) == len(saved_base)
assert len(saved_labels["y_train"]) == len(saved_train_idx)
assert len(saved_labels["y_validation"]) == len(saved_val_idx)
assert len(saved_labels["y_test"]) == len(saved_test_idx)
assert saved_schema["feature_columns"] == common_data.feature_columns
assert saved_base.columns.tolist() == [
    "TransactionID",
    "isFraud",
    *common_data.feature_columns,
]

print("Ortak Parquet, split, target/ID ve feature şeması doğrulandı.")
print("Ortak taban shape:", saved_base.shape)
print("Train/validation/test:", len(saved_train_idx), len(saved_val_idx), len(saved_test_idx))

Ortak Parquet, split, target/ID ve feature şeması doğrulandı.
Ortak taban shape: (590540, 425)
Train/validation/test: 413378 88581 88581


## Sonraki görevlerde cache nasıl kullanılacak?

Görev 2 veya Görev 3 ilk kez bir profile ihtiyaç duyduğunda:

```python
from common_preprocessing import get_or_create_profile_cache

linear_data = get_or_create_profile_cache("logistic_regression")
tree_data = get_or_create_profile_cache("random_forest")
catboost_data = get_or_create_profile_cache("catboost")
```

Fonksiyon önce cache'i kontrol eder. Cache varsa doğrudan yükler; yoksa ortak Parquet'ten ilgili profili bir kez üretir ve sonraki kullanımlar için saklar.

- Logistic Regression → `linear` sparse cache
- Decision Tree / Random Forest / XGBoost → paylaşılan `sklearn_tree` sparse cache
- LightGBM → native kategorik Parquet cache
- CatBoost → native kategorik Parquet cache

# Görev 1 tamamlanma koşulları

- Ortak left join, split ve train-temelli feature politikası uygulandı.
- Temizlenmiş ortak Parquet tabanı bir kez kaydedildi.
- Split, target/ID, feature şeması ve kalite raporları kaydedildi.
- Kaydedilen dosyalar yeniden açılarak doğrulandı.
- Kullanılmayan model profilleri boşuna preprocess edilmedi.
- Modele özel cache'ler ihtiyaç halinde bir kez üretilecek şekilde tanımlandı.
- Model eğitilmedi ve threshold seçilmedi.